Rectangular-cavity mDM overlap and power calculation.

This is the rectangular counterpart of Candpower.ipynb.  It
implements the uploaded analytic TE and TM fields and evaluates their overlap
with the centered symmetric-gauge source for a uniform magnetic field.


In [1]:
import numpy as np
from numpy import pi
from numpy.polynomial.legendre import leggauss
import matplotlib.pyplot as plt

## Set variables, modes, effective current J(x,y)

In [2]:
# Physical constants (set to 1 for dimensionless calculations)
epsilon0 = 1.0
mu0 = 1.0
c = 1.0 / np.sqrt(epsilon0 * mu0)

eps = 1e-20

# Rectangular cavity dimensions and physical parameters
a_raw = 0.419       # x dimension (m)
b_raw = 0.419       # y dimension (m)
d_raw = 1.014       # z dimension (m)
B_raw = 7.6         # magnetic field (T)
ro_dm_raw = 0.45    # dark-matter density (GeV/cc)

# Natural-unit conversions, following the original notebook
ro_dm = ro_dm_raw / 1.3e41
B = B_raw * 1.96e-16
Beta = 1.0
Q = 45000
a = a_raw * 5.0677e15
b = b_raw * 5.0677e15
d = d_raw * 5.0677e15
V = a * b * d

e_m = 4e-19

# Rectangular mode ranges: TE has m,n >= 0 (not both zero), p >= 1;
# TM has m,n >= 1, p >= 0.
m_max = 6
n_max = 6
p_max = 6

# For B0 parallel to z, use the centered symmetric gauge
# A0 = B0/2 [-(y-b/2) xhat + (x-a/2) yhat].
# Pulling out B0*L_ref/2 gives the dimensionless source below.
L_ref = np.sqrt(a * b)

quadrature_order = 24
nodes, weights = leggauss(quadrature_order)
nodes = 0.5 * (nodes + 1.0)
weights = 0.5 * weights

u = nodes[:, None, None]       # x/a
v = nodes[None, :, None]       # y/b
w = nodes[None, None, :]       # z/d
weights_3d = (
    weights[:, None, None]
    * weights[None, :, None]
    * weights[None, None, :]
)
zero_field = np.zeros((quadrature_order,) * 3, dtype=complex)


def volume_integral(values):
    """Integrate an array over the physical rectangular volume."""
    return V * np.sum(values * weights_3d)


def J_x(v_coord):
    """x component of the normalized mDM source."""
    return -b * (v_coord - 0.5) / L_ref


def J_y(u_coord):
    """y component of the normalized mDM source."""
    return a * (u_coord - 0.5) / L_ref


def J_z(w_coord):
    """z component of the normalized mDM source."""
    return 0.0 * w_coord


Jx = J_x(v)
Jy = J_y(u)
Jz = J_z(w)
J2_den = float(
    np.real(volume_integral(np.abs(Jx) ** 2 + np.abs(Jy) ** 2 + np.abs(Jz) ** 2))
)

## TE MODES COMPUTATION

In [3]:
resultsTE = []

# Loop over all TE combinations
for m in range(0, m_max + 1):
    for n in range(0, n_max + 1):
        if m == 0 and n == 0:
            continue

        for p in range(1, p_max + 1):
            mode_str = f"{m}{n}{p}"

            try:
                # Wavenumbers and resonant frequency
                kx = m * pi / a
                ky = n * pi / b
                kz = p * pi / d
                kc2 = kx**2 + ky**2
                omega = c * np.sqrt(kc2 + kz**2)
                mass = omega / 2.0
                mass_eV = mass * 1e9

                # First define the electric and magnetic mode components.
                # The arbitrary TE normalization H0 is set to one.
                def E_TE_x(u_coord, v_coord, w_coord):
                    return (
                        -1j
                        * omega
                        * mu0
                        * ky
                        / kc2
                        * np.cos(m * pi * u_coord)
                        * np.sin(n * pi * v_coord)
                        * np.sin(p * pi * w_coord)
                    )

                def E_TE_y(u_coord, v_coord, w_coord):
                    return (
                        1j
                        * omega
                        * mu0
                        * kx
                        / kc2
                        * np.sin(m * pi * u_coord)
                        * np.cos(n * pi * v_coord)
                        * np.sin(p * pi * w_coord)
                    )

                def E_TE_z(u_coord, v_coord, w_coord):
                    return zero_field

                def H_TE_x(u_coord, v_coord, w_coord):
                    return (
                        kx
                        * kz
                        / kc2
                        * np.sin(m * pi * u_coord)
                        * np.cos(n * pi * v_coord)
                        * np.cos(p * pi * w_coord)
                    )

                def H_TE_y(u_coord, v_coord, w_coord):
                    return (
                        ky
                        * kz
                        / kc2
                        * np.cos(m * pi * u_coord)
                        * np.sin(n * pi * v_coord)
                        * np.cos(p * pi * w_coord)
                    )

                def H_TE_z(u_coord, v_coord, w_coord):
                    return (
                        np.cos(m * pi * u_coord)
                        * np.cos(n * pi * v_coord)
                        * np.sin(p * pi * w_coord)
                    )

                Ex = E_TE_x(u, v, w)
                Ey = E_TE_y(u, v, w)
                Ez = E_TE_z(u, v, w)
                Hx = H_TE_x(u, v, w)
                Hy = H_TE_y(u, v, w)
                Hz = H_TE_z(u, v, w)

                E2_den = float(
                    np.real(
                        volume_integral(
                            np.abs(Ex) ** 2 + np.abs(Ey) ** 2 + np.abs(Ez) ** 2
                        )
                    )
                )
                H2_den = float(
                    np.real(
                        volume_integral(
                            np.abs(Hx) ** 2 + np.abs(Hy) ** 2 + np.abs(Hz) ** 2
                        )
                    )
                )

                numerator_integral = volume_integral(
                    np.conj(Ex) * Jx + np.conj(Ey) * Jy + np.conj(Ez) * Jz
                )

                if E2_den != 0:
                    # Same C convention used in the original Candpower notebook.
                    C = np.abs(numerator_integral) ** 2 / (E2_den * V)
                    C_normalized = np.abs(numerator_integral) ** 2 / (E2_den * J2_den)

                    if C < 1e-24:
                        C = 0.0
                        C_normalized = 0.0

                    # P = beta/(1+beta) e_m^4 B^2 Q L_ref^2 rho^2 V C /(8m^5)
                    P_mc = (
                        (Beta / (1.0 + Beta))
                        * B**2
                        * Q
                        * L_ref**2
                        * V
                        * C
                        * ro_dm**2
                        / (8.0 * mass**5)
                        / 4.11e-15
                    )
                    P_W = P_mc * e_m**4
                else:
                    C = np.nan
                    C_normalized = np.nan
                    P_W = np.nan
                    P_mc = np.nan

                # Stored-energy checks, analogous to the original notebook.
                mode_coefficient = numerator_integral / E2_den
                E2_stored = epsilon0 / 2.0 * np.abs(mode_coefficient) ** 2 * E2_den
                H2_stored = mu0 / 2.0 * np.abs(mode_coefficient) ** 2 * H2_den
                check1 = epsilon0 * E2_den / (mu0 * H2_den)
                check2 = E2_stored / H2_stored if H2_stored != 0 else np.nan

                omega_MHz = omega * 2.41799e17

                resultsTE.append(
                    (
                        m,
                        n,
                        p,
                        omega_MHz,
                        mass_eV,
                        C,
                        C_normalized,
                        P_W,
                        P_mc,
                        check1,
                        check2,
                    )
                )

            except Exception as error:
                print(f"  Error for mode {mode_str}: {error}")
                resultsTE.append(
                    (m, n, p, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)
                )

# Print TE table
print("\n" + "=" * 150)
print(
    f"{'m':<2} {'n':<2} {'p':<2} {'omega (MHz)':<15} {'mass (eV)':<15} "
    f"{'C':<15} {'C normalized':<15} {'Power (W)':<15} "
    f"{'Power/e_m^4':<15} {'E/H check':<15} {'stored check':<15}"
)
print("=" * 150)
for m, n, p, frequency, mass_value, C, Cn, power, power_coefficient, check1, check2 in resultsTE:
    print(
        f"{m:<2} {n:<2} {p:<2} {frequency:<15.6e} {mass_value:<15.6e} "
        f"{C:<15.6e} {Cn:<15.6e} {power:<15.6e} "
        f"{power_coefficient:<15.6e} {check1:<15.6e} {check2:<15.6e}"
    )
print("=" * 150)


# %%
# Minimum threshold ADMX (10^-22), optim
# Minimum e_m then of order 10^-19


# %% [markdown]


m  n  p  omega (MHz)     mass (eV)       C               C normalized    Power (W)       Power/e_m^4     E/H check       stored check   
0  1  1  3.870891e+02    8.004356e-07    0.000000e+00    0.000000e+00    0.000000e+00    0.000000e+00    1.000000e+00    1.000000e+00   
0  1  2  4.641088e+02    9.596995e-07    0.000000e+00    0.000000e+00    0.000000e+00    0.000000e+00    1.000000e+00    1.000000e+00   
0  1  3  5.697911e+02    1.178233e-06    0.000000e+00    0.000000e+00    0.000000e+00    0.000000e+00    1.000000e+00    1.000000e+00   
0  1  4  6.911099e+02    1.429100e-06    0.000000e+00    0.000000e+00    0.000000e+00    0.000000e+00    1.000000e+00    1.000000e+00   
0  1  5  8.211638e+02    1.698030e-06    0.000000e+00    0.000000e+00    0.000000e+00    0.000000e+00    1.000000e+00    1.000000e+00   
0  1  6  9.563958e+02    1.977667e-06    0.000000e+00    0.000000e+00    0.000000e+00    0.000000e+00    1.000000e+00    1.000000e+00   
0  2  1  7.306113e+02    1.510782e-06   

## TM MODES COMPUTATION

In [4]:
# %%
resultsTM = []

# Loop over all TM combinations
for m in range(1, m_max + 1):
    for n in range(1, n_max + 1):
        for p in range(0, p_max + 1):
            mode_str = f"{m}{n}{p}"
            print(f"Computing TM mode {mode_str}...")

            try:
                # Wavenumbers and resonant frequency
                kx = m * pi / a
                ky = n * pi / b
                kz = p * pi / d
                kc2 = kx**2 + ky**2
                omega = c * np.sqrt(kc2 + kz**2)

                # The arbitrary TM normalization E0 is set to one.
                def E_TM_x(u_coord, v_coord, w_coord):
                    return (
                        -kx
                        * kz
                        / kc2
                        * np.cos(m * pi * u_coord)
                        * np.sin(n * pi * v_coord)
                        * np.sin(p * pi * w_coord)
                    )

                def E_TM_y(u_coord, v_coord, w_coord):
                    return (
                        -ky
                        * kz
                        / kc2
                        * np.sin(m * pi * u_coord)
                        * np.cos(n * pi * v_coord)
                        * np.sin(p * pi * w_coord)
                    )

                def E_TM_z(u_coord, v_coord, w_coord):
                    return (
                        np.sin(m * pi * u_coord)
                        * np.sin(n * pi * v_coord)
                        * np.cos(p * pi * w_coord)
                    )

                def H_TM_x(u_coord, v_coord, w_coord):
                    return (
                        -1j
                        * omega
                        * epsilon0
                        * ky
                        / kc2
                        * np.sin(m * pi * u_coord)
                        * np.cos(n * pi * v_coord)
                        * np.cos(p * pi * w_coord)
                    )

                def H_TM_y(u_coord, v_coord, w_coord):
                    return (
                        1j
                        * omega
                        * epsilon0
                        * kx
                        / kc2
                        * np.cos(m * pi * u_coord)
                        * np.sin(n * pi * v_coord)
                        * np.cos(p * pi * w_coord)
                    )

                def H_TM_z(u_coord, v_coord, w_coord):
                    return zero_field

                Ex = E_TM_x(u, v, w)
                Ey = E_TM_y(u, v, w)
                Ez = E_TM_z(u, v, w)
                Hx = H_TM_x(u, v, w)
                Hy = H_TM_y(u, v, w)
                Hz = H_TM_z(u, v, w)

                E2_den = float(
                    np.real(
                        volume_integral(
                            np.abs(Ex) ** 2 + np.abs(Ey) ** 2 + np.abs(Ez) ** 2
                        )
                    )
                )
                H2_den = float(
                    np.real(
                        volume_integral(
                            np.abs(Hx) ** 2 + np.abs(Hy) ** 2 + np.abs(Hz) ** 2
                        )
                    )
                )

                overlap_integral = volume_integral(
                    np.conj(Ex) * Jx + np.conj(Ey) * Jy + np.conj(Ez) * Jz
                )

                overlap = overlap_integral / E2_den
                C_mDM = np.abs(overlap_integral) ** 2 / (E2_den * V)
                C_normalized = np.abs(overlap_integral) ** 2 / (E2_den * J2_den)

                if C_mDM < 1e-24:
                    C_mDM = 0.0
                    C_normalized = 0.0

                # Uniform z-polarized form factor, useful as an axion-mode check.
                uniform_z_integral = volume_integral(Ez)
                form_factor = np.abs(uniform_z_integral) ** 2 / (E2_den * V)
                check = epsilon0 * E2_den / (mu0 * H2_den)

                resultsTM.append(
                    (m, n, p, overlap, C_mDM, C_normalized, form_factor, check)
                )

            except Exception as error:
                print(f"  Error for mode {mode_str}: {error}")
                resultsTM.append(
                    (m, n, p, np.nan, np.nan, np.nan, np.nan, np.nan)
                )

# Print TM table
print("\n" + "=" * 125)
print(
    f"{'m':<2} {'n':<2} {'p':<2} {'Overlap':<20} {'C mDM':<15} "
    f"{'C normalized':<15} {'Uniform-z FF':<15} {'E/H check':<15}"
)
print("=" * 125)
for m, n, p, overlap, C_mDM, Cn, form_factor, check in resultsTM:
    print(
        f"{m:<2} {n:<2} {p:<2} {overlap!s:<20} {C_mDM:<15.6e} "
        f"{Cn:<15.6e} {form_factor:<15.6e} {check:<15.6e}"
    )
print("=" * 125)

Computing TM mode 110...
Computing TM mode 111...
Computing TM mode 112...
Computing TM mode 113...
Computing TM mode 114...
Computing TM mode 115...
Computing TM mode 116...
Computing TM mode 120...
Computing TM mode 121...
Computing TM mode 122...
Computing TM mode 123...
Computing TM mode 124...
Computing TM mode 125...
Computing TM mode 126...
Computing TM mode 130...
Computing TM mode 131...
Computing TM mode 132...
Computing TM mode 133...
Computing TM mode 134...
Computing TM mode 135...
Computing TM mode 136...
Computing TM mode 140...
Computing TM mode 141...
Computing TM mode 142...
Computing TM mode 143...
Computing TM mode 144...
Computing TM mode 145...
Computing TM mode 146...
Computing TM mode 150...
Computing TM mode 151...
Computing TM mode 152...
Computing TM mode 153...
Computing TM mode 154...
Computing TM mode 155...
Computing TM mode 156...
Computing TM mode 160...
Computing TM mode 161...
Computing TM mode 162...
Computing TM mode 163...
Computing TM mode 164...
